# SPARQL

This document contains a short introduction to [SPARQL](https://www.w3.org/TR/sparql11-query/) using [rudof](https://rudof-project.github.io/).


## Preliminaries

The library is available as `pyrudof`.

In [ ]:
!pip install pyrudof

The main entry point if a class called `Rudof` through which most of the functionality is provided.

In [1]:
from pyrudof import Rudof, RudofConfig

In order to initialize that class, it is possible to pass a RudofConfig instance which contains configuration parameters for customization.

In [2]:
rudof = Rudof(RudofConfig())

## SPARQL queries with local RDF data

As SPARQL is an RDF query language, we first need some RDF data.

rudof can run queries against local RDF data or RDF that is available through some SPARQL endpoint.

Let's start with some local RDF data.

In [3]:
rdf = """
prefix :    <http://example.org/>
prefix xsd: <http://www.w3.org/2001/XMLSchema#>

:alice a :Person ;
 :name      "Alice"                ;
 :birthDate "2005-03-01"^^xsd:date ;
 :worksFor  :acme                   .
:bob a :Person   ;
 :name      "Robert Smith"         ;
 :birthDate "2003-01-02"^^xsd:date ;
 :worksFor  :acme  .
:acme a :Company ;
 :name "Acme Inc." .
"""
rudof.read_data(rdf)

We can run SPARQL queries as follows:

In [4]:
query = """
PREFIX : <http://example.org/>

SELECT ?person ?name ?date ?company WHERE {
  ?person a          :Person ;
          :name      ?name   ;
          :birthDate ?date   ;
          :worksFor  ?c   .
  ?c      :name      ?company .
}
"""

rudof.read_query(query)
rudof.run_query()
results = rudof.serialize_query_results()

Show the results:

In [5]:
print(results)

╭───┬─────────┬────────────────┬───────────────────────────────────────────────────────┬─────────────╮
│   │ ?person │ ?name          │ ?date                                                 │ ?company    │
├───┼─────────┼────────────────┼───────────────────────────────────────────────────────┼─────────────┤
│ 1 │ :bob    │ "Robert Smith" │ "2003-01-02"^^<http://www.w3.org/2001/XMLSchema#date> │ "Acme Inc." │
├───┼─────────┼────────────────┼───────────────────────────────────────────────────────┼─────────────┤
│ 2 │ :alice  │ "Alice"        │ "2005-03-01"^^<http://www.w3.org/2001/XMLSchema#date> │ "Acme Inc." │
╰───┴─────────┴────────────────┴───────────────────────────────────────────────────────┴─────────────╯



In [6]:
rudof.reset_all()

## SPARQL queries with an endpoint

It is also possible to run SPARQL queries against SPARQL endpoints.

SPARQL endpoints are usually identified by an IRI.

rudof contains a list of popular SPARQL endpoints which can also be identified by name, like wikidata, whose IRI is: https://query.wikidata.org/sparql

In [7]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
SELECT ?person ?occupation WHERE {
    ?p wdt:P31 wd:Q5 ;
          wdt:P106 ?o ;
          rdfs:label ?person ;
          wdt:P19 wd:Q14317 .
  ?o rdfs:label ?occupation
  FILTER (lang(?person) = "en" && lang(?occupation) = "en")
}
LIMIT 10
"""

In [8]:
rudof.read_data(endpoint="https://query.wikidata.org/sparql")

In [9]:
rudof.read_query(query)
rudof.run_query()
results = rudof.serialize_query_results()

In [10]:
print(results)

╭────┬──────────────────────────────────────────┬────────────────────────╮
│    │ ?person                                  │ ?occupation            │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 1  │ "Elena Fernández"en                      │ "actor"en              │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 2  │ "Luciano García del Real"en              │ "writer"en             │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 3  │ "Mercedes Neuschäfer-Carlón"en           │ "writer"en             │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 4  │ "Javier Almuzara"en                      │ "writer"en             │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 5  │ "Leticia Sánchez Ruiz"en                 │ "writer"en             │
├────┼──────────────────────────────────────────┼────────────────────────┤
│ 6  │ "Alberto Rionda"en

In [11]:
rudof.reset_all()

## Federated queries

TBD

## CONSTRUCT queries

In [12]:
from pyrudof import QueryResultFormat

It is also possible to run SPARQL CONSTRUCT queries which can be used to return RDF results.

For example:

In [13]:
query = """
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX :    <http://example.org/>

CONSTRUCT {
   ?p a     :Person ;
      :name ?person ;
      :occupation ?occupation
} WHERE {
    ?p wdt:P31 wd:Q5 ;
          wdt:P106 ?o ;
          rdfs:label ?person ;
          wdt:P19 wd:Q14317 .
  ?o rdfs:label ?occupation
  FILTER (lang(?person) = "en" && lang(?occupation) = "en")
}
LIMIT 10
"""

We indicate rudof that we will use Wikidata's endpoint:

In [14]:
rudof.read_data(endpoint="https://query.wikidata.org/sparql")

In [15]:
rudof.read_query(query)
rudof.run_query()
result = rudof.serialize_query_results(QueryResultFormat.Turtle)

In [16]:
print(result)

@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sesame: <http://www.openrdf.org/schema/sesame#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix fn: <http://www.w3.org/2005/xpath-functions#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix hint: <http://www.bigdata.com/queryHints#> .
@prefix bd: <http://www.bigdata.com/rdf#> .
@prefix bds: <http://www.bigdata.com/rdf/search#> .
@prefix psn: <http://www.wikidata.org/prop/statement/value-normalized/> .
@prefix pqn: <http://www.wikidata.org/prop/qualifier/value-normalized/> .
@prefix prn: <http://www.wikidata.org/prop/reference/value-normalized/> .
@prefix mwapi: <https://www.mediawiki.org/ontology#API/> .
@prefix gas: <http://www.bigdata.com/rdf/gas#> .
@prefix wdsubgraph: <https://query.wikidata.org/subgraph/> .
@prefix wdt: <http://www.wikidata.or

## References



### Resources for learning SPARQL

*   [Learning SPARQL](https://learningsparql.com/) book and blog by Bob Ducharne.
*   [SPARQL tutorial](https://www.wikidata.org/wiki/Wikidata:SPARQL_tutorial) from Wikidata

### Lists of SPARQL endpoints

*   [SPARQL endpoints](https://www.wikidata.org/wiki/Wikidata:Lists/SPARQL_endpoints) collected in Wikidata
*   [SPARQL endpoints](https://www.w3.org/wiki/SparqlEndpoints) collected at W3C.

